# Score the frozen test sets and export paper results

Run after all 75 student fits and their audits have completed. This checks the entire matrix before evaluating the final cohorts, publishes per-image logits, predictions and confusion matrices and writes accuracy.csv, paired.json, resources.csv and reuse.csv to the Kaggle output folder.

**Setup:** GPU T4 ×2, Internet enabled, all three project inputs attached, and the Kaggle secret `HF_TOKEN` enabled for this notebook. Keep the token out of code. The repository and worker settings are filled in. Use **Run All**. The mounted files and split are pinned by their hashes. See KAGGLE_RUN_INSTRUCTIONS.md for links and recovery.

In [ ]:
HF_REPO = "Shanmuk4622/kd-crop-budget-4w-2026"
TINY_KAGGLE_SOURCE = "shanmuk4622/tinyimagenet200"
GROUP = "all"
NUM_WORKERS = 4
VERSIONS = {}  # File hashes pin each Kaggle input automatically
INPUT_PATHS = {}  # Usually detected. To override: {"cifar100": "/kaggle/input/.../dataset-cifar100-python"}
SCRATCH = "/kaggle/temp/kd_view_v2"  # Can also use /kaggle/working/kd_view_v2

### Runtime dependencies

The final cell starts the stage in the qualified software environment. On a newer Kaggle image it installs that environment in scratch space.

In [ ]:
import sys
if sys.version.split()[0] == '3.12.13':
    import importlib.util, subprocess, sys
    packages = {"huggingface_hub":"huggingface_hub", "timm":"timm", "pynvml":"nvidia-ml-py", "scipy":"scipy", "filelock":"filelock"}
    missing = [package for module,package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable,"-m","pip","install","--disable-pip-version-check",*missing])

### Embedded experiment code

The self-contained bundle verifies its digest before import.

In [ ]:
import base64, hashlib, io, os, sys, tempfile, zipfile
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "67df1731479c785aa54c9535e210d644d1a6efe9df51eb455684e7418dbc4e2d"
runtime = Path(tempfile.gettempdir()) / "kd-runtime-67df1731479c78"
runtime.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    for name in bundle.namelist():
        dest = runtime / name
        value = bundle.read(name)
        if dest.exists() and dest.read_bytes() != value:
            raise RuntimeError("Incompatible extracted runtime. Restart in a clean session.")
        if not dest.exists(): dest.write_bytes(value)
sys.path.insert(0,str(runtime))
print("Verified experiment runtime", "67df1731479c78")

### Locate Tiny ImageNet

Select the dataset folder inside the Kaggle mount before qualification, training or scoring.

In [ ]:
"""Resolve the Tiny ImageNet mount before entering the published experiment runtime."""

from pathlib import Path


def tiny_input_path(source, override=None, base=Path("/kaggle/input")):
    owner, slug = source.split("/")
    if not owner or not slug or owner in (".", "..") or slug in (".", ".."):
        raise ValueError("Set TINY_KAGGLE_SOURCE to the attached Kaggle dataset's owner/slug.")
    base = base.resolve()
    if override:
        root = Path(override).resolve()
        if not root.is_dir() or not root.is_relative_to(base) or slug not in root.parts:
            raise ValueError(f"Set INPUT_PATHS['tiny_imagenet200'] to the mounted folder for {source}.")
    else:
        owned = base / "datasets" / owner / slug
        roots = [p for p in (owned, base / slug) if p.is_dir()]
        if not roots:
            roots = [p for p in base.rglob(slug) if p.is_dir()]
        if len(roots) != 1:
            raise ValueError(f"Attach {source} or specify its exact folder in INPUT_PATHS. Found {roots}.")
        root = roots[0]

    # Check the canonical layout before walking a mount with 120,000 images.
    for folder in (root / "tiny-imagenet-200", root):
        if ((folder / "wnids.txt").is_file() and (folder / "train").is_dir()
                and (folder / "val").is_dir() and not (folder / "tiny-imagenet-200").is_dir()):
            return str(folder)
    markers = [p for p in root.rglob("wnids.txt") if p.is_file() and "__MACOSX" not in p.parts]
    if not markers:
        # The experiment reader also supports the original, unopened Stanford ZIP.
        return str(root)

    candidates = []
    for marker in markers:
        folder = marker.parent
        # The reader must see a single class list beneath the selected folder.
        if any(other != marker and other.is_relative_to(folder) for other in markers):
            continue
        if (folder / "train").is_dir() and (folder / "val").is_dir():
            candidates.append(folder)
    if len(candidates) != 1:
        raise ValueError(
            f"Found {len(candidates)} self-contained Tiny ImageNet folders in {root}: {candidates}. "
            "Select one complete folder in INPUT_PATHS['tiny_imagenet200'], or attach the original Stanford ZIP."
        )
    return str(candidates[0])


RESOLVED_PATHS = dict(INPUT_PATHS)
RESOLVED_PATHS["tiny_imagenet200"] = tiny_input_path(TINY_KAGGLE_SOURCE, INPUT_PATHS.get("tiny_imagenet200"))
print("Tiny ImageNet input:", RESOLVED_PATHS["tiny_imagenet200"])


### Stage code

In [ ]:
stage_program = r'''
from kaggle_inputs import SOURCES
from kaggle_run import runtime_id

actual = runtime_id()
differences = [f"{key}: expected {value}, found {actual.get(key)}"
               for key, value in expected_software.items() if actual.get(key) != value]
if differences:
    raise RuntimeError("Software mismatch after launch: " + ", ".join(differences))
SOURCES["tiny_imagenet200"] = TINY_KAGGLE_SOURCE

from kaggle_run import check_settings, session, qualify, train
from kaggle_results import release
check_settings(HF_REPO, RESOLVED_PATHS, VERSIONS)
with session(HF_REPO, SCRATCH, 'results') as (hub, base, output):
    result = release(hub,base,output,RESOLVED_PATHS,VERSIONS)
print(result)
'''

### Execute this stage

In [ ]:
"""Run each Kaggle stage with the software recorded by the completed preflight."""

import importlib.util
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
from pathlib import Path


QUALIFIED_SOFTWARE = {
    "python": "3.12.13", "torch": "2.10.0+cu128", "torchvision": "0.25.0+cu128",
    "cuda": "12.8", "cudnn": 91002, "numpy": "2.0.2", "pillow": "11.3.0", "timm": "1.0.26",
}


def compatible_kernel():
    if sys.version.split()[0] != QUALIFIED_SOFTWARE["python"]:
        return False
    try:
        from kaggle_run import runtime_id
        actual = runtime_id()
    except (ImportError, OSError, RuntimeError):
        return False
    return all(actual.get(k) == v for k, v in QUALIFIED_SOFTWARE.items())


def install_worker_python(root, env):
    root.mkdir(parents=True, exist_ok=True)
    tools = root / "tools"
    uv = tools / "bin" / "uv"
    if not uv.is_file():
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--upgrade",
                        "--target", str(tools), "uv"], check=True)
    env.update(UV_PYTHON_INSTALL_DIR=str(root / "interpreters"), UV_CACHE_DIR=str(root / "cache"),
               UV_LINK_MODE="hardlink")
    venv = root / "venv"
    python = venv / "bin" / "python"
    if not python.exists():
        print("Installing Python 3.12.13 in Kaggle scratch space.", flush=True)
        subprocess.run([uv, "venv", "--managed-python", "--python", QUALIFIED_SOFTWARE["python"], venv],
                       env=env, check=True)
    wheels = []
    for package in ("torch", "torchvision"):
        version = QUALIFIED_SOFTWARE[package].replace("+", "%2B")
        wheel = f"{package}-{version}-cp312-cp312-manylinux_2_28_x86_64.whl"
        wheels.append(f"https://download.pytorch.org/whl/cu128/{wheel}")
    requirements = [*wheels, "numpy==2.0.2", "pillow==11.3.0", "timm==1.0.26",
                    "huggingface_hub", "requests", "scipy", "filelock", "nvidia-ml-py", "psutil"]
    print("Checking the preflight packages and installing any missing CUDA 12.8 wheels.", flush=True)
    subprocess.run([uv, "pip", "install", "--python", python, *requirements], env=env, check=True)
    return python


def launch_stage(runtime, program, settings, role):
    settings = {**settings, "expected_software": QUALIFIED_SOFTWARE}
    if compatible_kernel():
        exec(compile(program, f"<{role}>", "exec"), dict(settings))
        return
    print("Preparing the preflight Python 3.12.13 / CUDA 12.8 environment. "
          "The first run downloads the GPU packages.", flush=True)
    env = os.environ.copy()
    for key in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        env.pop(key, None)
    env["PYTHONNOUSERSITE"] = "1"
    # Keep the mounted driver visible. Load CUDA user libraries from the qualified wheels.
    libs = env.get("LD_LIBRARY_PATH", "").split(os.pathsep)
    env["LD_LIBRARY_PATH"] = os.pathsep.join(p for p in libs if p and "site-packages" not in p
                                              and not ("/cuda" in p and not p.endswith("/compat")))
    root = Path("/kaggle/temp") / f"kd-{role}-python31213-cu128"
    python = install_worker_python(root, env)
    with tempfile.TemporaryDirectory(dir=root) as tmp:
        folder = Path(tmp)
        # Share only Kaggle's Python secrets client. It obtains credentials in the child.
        for module in ("kaggle_web_client", "kaggle_secrets"):
            spec = importlib.util.find_spec(module)
            if not spec or not spec.origin or not spec.origin.endswith(".py"):
                raise RuntimeError(f"Kaggle's {module} module is unavailable in this session.")
            shutil.copyfile(spec.origin, folder / (module + ".py"))
        script = folder / "stage.py"
        script.write_text("import sys, json\n"
                          + f"sys.path.insert(0, {str(runtime)!r})\n"
                          + "globals().update(json.loads(sys.argv[1]))\n" + program, encoding="utf-8")
        # Settings contain paths and worker options. Tokens are read through Kaggle Secrets.
        command = [str(python), "-u", str(script), json.dumps(settings)]
        print(f"Starting {role} in the qualified environment.", flush=True)
        with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, errors="replace", start_new_session=True) as proc:
            def forward(signum, frame):
                if proc.poll() is None:
                    proc.send_signal(signum)
            previous = {s: signal.signal(s, forward) for s in (signal.SIGINT, signal.SIGTERM)}
            try:
                for line in proc.stdout:
                    print(line, end="", flush=True)
                if proc.wait():
                    raise RuntimeError(f"{role} exited with status {proc.returncode}. See its output above.")
            finally:
                for s, handler in previous.items():
                    signal.signal(s, handler)

settings = {'HF_REPO': HF_REPO, 'SCRATCH': SCRATCH, 'TINY_KAGGLE_SOURCE': TINY_KAGGLE_SOURCE, 'RESOLVED_PATHS': RESOLVED_PATHS, 'VERSIONS': VERSIONS}
launch_stage(runtime, stage_program, settings, 'results')